In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd
import numpy as np

In [2]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
#teams.sort()
#print(teams)

In [3]:
replace_dict = {"SFO":"SF", "NOS":"NO", "GBP":"GB", "KCC": "KC", "NE": "NEP", "NOS": "NO", "SFO":"SF"}
clean_teams = [replace_dict.get(item,item) for item in teams]
#print(clean_teams)

In [4]:
# df_odds = dict(zip(clean_teams, odds))
# #print(df_odds)
# print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

In [18]:
df_fd = pd.read_csv("data/nfl/FanDuel-NFL-2021 ET-10 ET-03 ET-64729-players-list.csv")

In [19]:
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,64729-6498,QB,Tom,Tom Brady,Brady,29.160001,3.0,16000,TB@NE,TB,NE,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
1,64729-90584,QB,Mac,Mac Jones,Jones,12.426666,3.0,14500,TB@NE,NE,TB,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
2,64729-55635,WR,Chris,Chris Godwin,Godwin,16.600000,3.0,14000,TB@NE,TB,NE,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
3,64729-32384,WR,Mike,Mike Evans,Evans,13.500000,3.0,13000,TB@NE,TB,NE,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
4,64729-64576,RB,Damien,Damien Harris,Harris,8.566667,3.0,12000,TB@NE,NE,TB,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX


In [20]:
df_fd.columns

Index(['Id', 'Position', 'First Name', 'Nickname', 'Last Name', 'FPPG',
       'Played', 'Salary', 'Game', 'Team', 'Opponent', 'Injury Indicator',
       'Injury Details', 'Tier', 'Unnamed: 14', 'Unnamed: 15',
       'Roster Position'],
      dtype='object')

In [21]:
df_fd.set_index('Id',inplace=True)

In [35]:
#df.drop(df.dropna(subset=['columnA']).index)
df_fd = df_fd.drop(df_fd.dropna(subset=['Injury Indicator']).index)
df_fd = df_fd[df_fd.Nickname != 'Cam Newton']
df_fd = df_fd[df_fd.Nickname != 'Garrett Gilbert']

df_fd.head()

,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
Id,,,,,,,,,,,,,,,,
64729-6498,QB,Tom,Tom Brady,Brady,29.160001,3.0,16000,TB@NE,TB,NE,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
64729-90584,QB,Mac,Mac Jones,Jones,12.426666,3.0,14500,TB@NE,NE,TB,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
64729-55635,WR,Chris,Chris Godwin,Godwin,16.600000,3.0,14000,TB@NE,TB,NE,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
64729-32384,WR,Mike,Mike Evans,Evans,13.500000,3.0,13000,TB@NE,TB,NE,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX
64729-64576,RB,Damien,Damien Harris,Harris,8.566667,3.0,12000,TB@NE,NE,TB,NaN,NaN,NaN,NaN,NaN,MVP - 1.5X Points/AnyFLEX


In [36]:
df_fd.to_csv('data/nfl/FDprojectionsSD.csv')

In [37]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter

optimizer = get_optimizer(Site.FANDUEL_SINGLE_GAME, Sport.FOOTBALL)
optimizer.load_players_from_csv("data/nfl/FDprojectionsSD.csv")
for lineup in optimizer.optimize(9, max_exposure=.3):
    print(lineup)
optimizer.export('data/nfl/fd_resultSD.csv')

 1. MVP     Tom Brady                     QB    TB             TB@NE    43.74          16000.0$  
 2. UTIL    Antonio Brown                 WR    TB             TB@NE    11.7           10000.0$  
 3. UTIL    Chris Godwin                  WR    TB             TB@NE    16.6           14000.0$  
 4. UTIL    Darwin Thompson               RB    TB             TB@NE    5.94           5500.0$   
 5. UTIL    Mac Jones                     QB    NE             TB@NE    12.427         14500.0$  

Fantasy Points 90.41
Salary 60000.00

 1. MVP     Tom Brady                     QB    TB             TB@NE    43.74          16000.0$  
 2. UTIL    Antonio Brown                 WR    TB             TB@NE    11.7           10000.0$  
 3. UTIL    Chris Godwin                  WR    TB             TB@NE    16.6           14000.0$  
 4. UTIL    Mike Evans                    WR    TB             TB@NE    13.5           13000.0$  
 5. UTIL    N'Keal Harry                  WR    NE             TB@NE    4.415  